In [1]:
import json
import glob
import re
import pandas as pd
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from macrogen import update_macro

In [2]:
RESULTS_DIR = '../results'
LABELS = ['Liberal', 'Neutral', 'Conservative']

In [3]:
results = glob.glob(RESULTS_DIR + '/**/YTIDEOLOGY/**/20000_cands*/**/test-1000.json', recursive=True)
results += glob.glob(RESULTS_DIR + '/**/YTIDEOLOGY/**/0_cands*/**/test-1000.json', recursive=True)
results

['../results/test/YTIDEOLOGY/test/125_shots/bertscore/20000_cands-deberta-large-mnli-recall/s0/LLAMA7B/test-1000.json',
 '../results/test/YTIDEOLOGY/test/125_shots/bertscore/20000_cands-deberta-large-mnli-recall/s0/MISTRAL/test-1000.json',
 '../results/test/YTIDEOLOGY/test/125_shots/bertscore/20000_cands-deberta-large-mnli-recall/s0/GPT4/test-1000.json',
 '../results/test/YTIDEOLOGY/test/4_shots/bertscore/20000_cands-deberta-large-mnli-recall/s0/LLAMA7B/test-1000.json',
 '../results/test/YTIDEOLOGY/test/4_shots/bertscore/20000_cands-deberta-large-mnli-recall/s0/MISTRAL/test-1000.json',
 '../results/test/YTIDEOLOGY/test/4_shots/bertscore/20000_cands-deberta-large-mnli-recall/s0/LLAMA13B/test-1000.json',
 '../results/test/YTIDEOLOGY/test/4_shots/bertscore/20000_cands-deberta-large-mnli-recall/s0/GPT4/test-1000.json',
 '../results/test/YTIDEOLOGY/test/36_shots/bertscore/20000_cands-deberta-large-mnli-recall/s0/LLAMA7B/test-1000.json',
 '../results/test/YTIDEOLOGY/test/36_shots/bertscore/2

In [4]:
test_set = pd.read_csv('../data/classification/yt_ideology/test.csv')
test_set['true'] = test_set['label'].map(lambda x : ['Liberal', 'Neutral', 'Conservative'][x])
test_set = test_set.set_index('Unnamed: 0')
test_set.head()

,text,label,slant,true
Unnamed: 0,,,,
26720,Tackling Africa's Industrialisation Challenge,1,-0.50000,Neutral
42818,Rep. Al Green - The Greatness of America Depen...,0,-1.00000,Liberal
10286,محمد بن زايد يصل قاعة الشعب الكبرى في العاصمة ...,1,-0.12000,Neutral
25376,Rep. Jeffries Dissects The Republican-Led Gove...,0,-1.00000,Liberal
1908,Eric Trump slams the FBI's decision in Clinton...,2,0.80723,Conservative


In [5]:
def sanitize_prediction(x):
    if 'neutral' in x.lower():
        return 'Neutral'
    if 'liberal' in x.lower():
        return 'Liberal'
    if 'conservative' in x.lower():
        return 'Conservative'
    return 'Neutral'

In [6]:
test_set = pd.read_csv('../data/classification/yt_ideology/test.csv')
test_set['true'] = test_set['label'].map(lambda x : ['Liberal', 'Neutral', 'Conservative'][x])
test_set = test_set.set_index('Unnamed: 0')

keys = []

for result in results:
    groups = re.search(
        r'results/test/YTIDEOLOGY/test/(?P<num_shots>[0-9]+)?_shots/.*?/[0-9]+?_cands.*?/s0/(?P<llm_name>.*)?/test-1000.json',
        result
    )

    llm_name = groups.group('llm_name')
    num_shots = groups.group('num_shots')

    print(llm_name, num_shots)

    with open(result) as f:
        js = json.load(f)

    key = f'{llm_name}_{num_shots}'
    keys.append(key)
    
    rows = []
    for res in js['results']:
        pred = sanitize_prediction(res['pred'])
        idx = res['Unnamed: 0']
        rows.append({
            key: pred,
            'idx': idx
        })
    df = pd.DataFrame(rows).set_index('idx')
    test_set = test_set.join(df)

LLAMA7B 125
MISTRAL 125
GPT4 125
LLAMA7B 4
MISTRAL 4
LLAMA13B 4
GPT4 4
LLAMA7B 36
MISTRAL 36
LLAMA13B 36
GPT4 36
LLAMA7B 12
MISTRAL 12
LLAMA13B 12
GPT4 12
LLAMA7B 8
MISTRAL 8
LLAMA13B 8
GPT4 8
LLAMA7B 0
MISTRAL 0
LLAMA13B 0
GPT4 0


In [7]:
df[key].value_counts()

GPT4_0
Neutral         426
Conservative    185
Liberal         139
Name: count, dtype: int64

In [8]:
def key_to_macro(k):
    llm_mapping = {
        'LLAMA7B': 'llamaseven',
        'LLAMA13B': 'llamathirteen',
        'GPT4': 'gpt',
        'MISTRAL': 'mistral'
    }
    shots_mapping = {
        '0': 'zero',
        '4': 'four',
        '8': 'eight',
        '12': 'twelve',
        '36': 'thirtysix',
        '125': 'onetwofive'
    }
    toks = k.split('_')
    return llm_mapping[toks[0]] + shots_mapping[toks[1]]

In [9]:
for key in keys:
    macro = f'{key_to_macro(key)}acc'
    print(key, '%.2f' % accuracy_score(test_set['true'], test_set[key]))
    update_macro(macro, '%.2f' % accuracy_score(test_set['true'], test_set[key]))

LLAMA7B_125 0.38
MISTRAL_125 0.34
GPT4_125 0.63
LLAMA7B_4 0.41
MISTRAL_4 0.35
LLAMA13B_4 0.47
GPT4_4 0.61
LLAMA7B_36 0.39
MISTRAL_36 0.33
LLAMA13B_36 0.41
GPT4_36 0.63
LLAMA7B_12 0.41
MISTRAL_12 0.34
LLAMA13B_12 0.44
GPT4_12 0.64
LLAMA7B_8 0.41
MISTRAL_8 0.34
LLAMA13B_8 0.46
GPT4_8 0.63
LLAMA7B_0 0.42
MISTRAL_0 0.33
LLAMA13B_0 0.48
GPT4_0 0.60


In [10]:
for key in keys:
    macro = f'{key_to_macro(key)}'
    precision, recall, fscore, support = precision_recall_fscore_support(test_set['true'], test_set[key], labels=LABELS)
    for idx, label in enumerate(LABELS):
        key = f'{macro}{label.lower()}precision'
        update_macro(key, '%.2f' % (precision[idx]))

        key = f'{macro}{label.lower()}recall'
        update_macro(key, '%.2f' % (recall[idx]))

/home/mharoon/miniconda3/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/mharoon/miniconda3/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/mharoon/miniconda3/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/mharoon/minicond